# 📦 BhashaLens — Notebook 3: Quantization & ONNX Export

Takes trained MarianMT models from Notebook 2 and:
1. Quantizes them INT8 for mobile (target: <30MB per model)
2. Converts to ONNX format for cross-platform inference
3. Validates quality (BLEU degradation <2 points)
4. Packages for Flutter app deployment

---

## 1. Setup & Dependencies

In [ ]:
%pip install -q transformers[torch] sentencepiece sacrebleu onnx onnxruntime optimum python-dotenv

In [ ]:
import os
import json
import time
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from transformers import MarianMTModel, MarianTokenizer
import onnx
import onnxruntime as ort
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import sacrebleu

print(f'ONNX Runtime version: {ort.__version__}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ── Local Paths ──
BASE_DIR = Path('./bhashalens_ml')

DATA_CLEANED = BASE_DIR / 'data' / 'cleaned'
MODELS_TRAINED = BASE_DIR / 'models' / 'trained'
MODELS_QUANTIZED = BASE_DIR / 'models' / 'quantized'
MODELS_ONNX = BASE_DIR / 'models' / 'onnx'
PACKAGES_DIR = BASE_DIR / 'models' / 'packages'
REPORTS_DIR = BASE_DIR / 'reports'

for d in [MODELS_QUANTIZED, MODELS_ONNX, PACKAGES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Trained models at: {MODELS_TRAINED}')


In [ ]:
# ── Configuration ──
EXPORT_CONFIG = {
    'quantization_method': 'int8',
    'max_bleu_degradation': 2.0,
    'target_size_mb': 30.0,
    'max_package_size_mb': 35.0,
    'onnx_opset': 13,
    'output_tolerance': 0.01,
    
    'inference_config': {
        'max_input_length': 512,
        'beam_size': 4,
        'length_penalty': 0.6,
        'no_repeat_ngram_size': 3,
        'early_stopping': True,
        'num_return_sequences': 1,
        'execution_providers': ['CPUExecutionProvider', 'NNAPIExecutionProvider'],
    },
    
    'bleu_thresholds': {
        'hi-en': 25.0, 'en-hi': 25.0,
        'mr-en': 20.0, 'en-mr': 20.0,
        'ta-en': 20.0, 'en-ta': 20.0,
        'gu-en': 20.0, 'en-gu': 20.0,
    },
    
    'model_version': '1.0.0',
}

# Discover trained models
available_models = []
if MODELS_TRAINED.exists():
    for d in sorted(MODELS_TRAINED.iterdir()):
        if d.is_dir() and (d / 'config.json').exists():
            available_models.append(d.name)

print(f'Available trained models: {available_models}')

# Select which to process
PAIRS_TO_EXPORT = available_models  # Export all available
print(f'Will export: {PAIRS_TO_EXPORT}')

## 2. Helper: BLEU Evaluation

In [ ]:
def evaluate_bleu(translate_fn, test_df: pd.DataFrame, n_samples: int = 500) -> float:
    """Evaluate BLEU score using a translation function."""
    sample = test_df.sample(n=min(n_samples, len(test_df)), random_state=42)
    
    hypotheses = []
    references = []
    
    for _, row in tqdm(sample.iterrows(), total=len(sample), desc='Evaluating BLEU'):
        translation = translate_fn(row['source'])
        hypotheses.append(translation)
        references.append(row['target'])
    
    result = sacrebleu.corpus_bleu(hypotheses, [references])
    return round(result.score, 2)


def get_model_size_mb(model_dir: Path) -> float:
    """Calculate total model size in MB."""
    total = sum(
        f.stat().st_size for f in model_dir.rglob('*')
        if f.is_file() and f.suffix in ['.bin', '.onnx', '.model', '.json', '.txt', '.safetensors']
    )
    return round(total / (1024 * 1024), 1)

print('Helper functions ready.')

## 3. INT8 Dynamic Quantization

In [ ]:
def quantize_model(lang_pair: str) -> dict:
    """Quantize a trained model using dynamic INT8 quantization."""
    print(f'\n{"="*60}')
    print(f'Quantizing: {lang_pair}')
    print(f'{"="*60}')
    
    model_dir = MODELS_TRAINED / lang_pair
    output_dir = MODELS_QUANTIZED / lang_pair
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load model
    print('Loading trained model...')
    tokenizer = MarianTokenizer.from_pretrained(str(model_dir))
    model = MarianMTModel.from_pretrained(str(model_dir))
    model.eval()
    
    original_size = get_model_size_mb(model_dir)
    print(f'Original model size: {original_size:.1f} MB')
    
    # Dynamic INT8 quantization
    print('Applying INT8 dynamic quantization...')
    quantized_model = torch.quantization.quantize_dynamic(
        model,
        {torch.nn.Linear},  # Quantize all Linear layers
        dtype=torch.qint8,
    )
    
    # Save quantized model
    # Note: We save the state_dict and reload to get proper serialization
    torch.save(quantized_model.state_dict(), output_dir / 'model_quantized.pt')
    tokenizer.save_pretrained(str(output_dir))
    
    # Also save the original model config for later use
    model.config.save_pretrained(str(output_dir))
    
    quantized_size = get_model_size_mb(output_dir)
    compression = (1 - quantized_size / original_size) * 100 if original_size > 0 else 0
    
    print(f'Quantized model size: {quantized_size:.1f} MB ({compression:.0f}% reduction)')
    
    # Check size constraint
    size_ok = quantized_size <= EXPORT_CONFIG['target_size_mb']
    print(f'Size check (<{EXPORT_CONFIG["target_size_mb"]}MB): {"✅ PASS" if size_ok else "❌ FAIL"}')
    
    return {
        'model': quantized_model,
        'tokenizer': tokenizer,
        'original_model': model,
        'original_size_mb': original_size,
        'quantized_size_mb': quantized_size,
        'compression_pct': compression,
        'size_ok': size_ok,
    }

print('Quantization function ready.')

In [ ]:
def validate_quantization(lang_pair: str, q_result: dict) -> dict:
    """Validate quantized model quality against original."""
    print(f'\nValidating quantization for {lang_pair}...')
    
    # Load test data
    test_path = DATA_CLEANED / lang_pair / 'test.tsv'
    if not test_path.exists():
        print(f'  No test data at {test_path}, skipping BLEU validation.')
        return {'bleu_validated': False}
    
    test_df = pd.read_csv(test_path, sep='\t')
    
    tokenizer = q_result['tokenizer']
    original_model = q_result['original_model']
    quantized_model = q_result['model']
    
    # Translation function for original
    def translate_original(text):
        inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = original_model.generate(**inputs, max_length=128, num_beams=4)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Translation function for quantized
    def translate_quantized(text):
        inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = quantized_model.generate(**inputs, max_length=128, num_beams=4)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Evaluate both
    print('  Evaluating original model BLEU...')
    original_bleu = evaluate_bleu(translate_original, test_df, n_samples=200)
    
    print('  Evaluating quantized model BLEU...')
    quantized_bleu = evaluate_bleu(translate_quantized, test_df, n_samples=200)
    
    degradation = original_bleu - quantized_bleu
    max_degradation = EXPORT_CONFIG['max_bleu_degradation']
    bleu_ok = degradation <= max_degradation
    
    print(f'\n  Original BLEU:  {original_bleu:.2f}')
    print(f'  Quantized BLEU: {quantized_bleu:.2f}')
    print(f'  Degradation:    {degradation:.2f} (max allowed: {max_degradation})')
    print(f'  BLEU check: {"✅ PASS" if bleu_ok else "❌ FAIL"}')
    
    if not bleu_ok:
        print(f'\n  ⚠️ BLEU degradation too high! Consider mixed INT8/FP16 precision.')
    
    return {
        'bleu_validated': True,
        'original_bleu': original_bleu,
        'quantized_bleu': quantized_bleu,
        'degradation': degradation,
        'bleu_ok': bleu_ok,
    }

print('Validation function ready.')

## 4. ONNX Conversion

In [ ]:
def convert_to_onnx(lang_pair: str) -> dict:
    """Convert trained model to ONNX format using Optimum."""
    print(f'\n{"="*60}')
    print(f'ONNX Conversion: {lang_pair}')
    print(f'{"="*60}')
    
    model_dir = MODELS_TRAINED / lang_pair
    onnx_dir = MODELS_ONNX / lang_pair
    onnx_dir.mkdir(parents=True, exist_ok=True)
    
    # Convert using Optimum
    print('Converting to ONNX with Optimum...')
    try:
        ort_model = ORTModelForSeq2SeqLM.from_pretrained(
            str(model_dir),
            export=True,
        )
        
        # Save ONNX model
        ort_model.save_pretrained(str(onnx_dir))
        
        # Also save tokenizer
        tokenizer = MarianTokenizer.from_pretrained(str(model_dir))
        tokenizer.save_pretrained(str(onnx_dir))
        
        onnx_size = get_model_size_mb(onnx_dir)
        print(f'ONNX model size: {onnx_size:.1f} MB')
        
        return {
            'ort_model': ort_model,
            'tokenizer': tokenizer,
            'onnx_dir': onnx_dir,
            'size_mb': onnx_size,
            'success': True,
        }
        
    except Exception as e:
        print(f'Optimum export failed: {e}')
        print('\nFalling back to manual ONNX export...')
        return manual_onnx_export(lang_pair, model_dir, onnx_dir)


def manual_onnx_export(lang_pair: str, model_dir: Path, onnx_dir: Path) -> dict:
    """Manual ONNX export fallback."""
    tokenizer = MarianTokenizer.from_pretrained(str(model_dir))
    model = MarianMTModel.from_pretrained(str(model_dir))
    model.eval()
    
    # Export encoder
    dummy_input = tokenizer('Hello, how are you?', return_tensors='pt', padding=True)
    
    encoder_path = onnx_dir / 'encoder_model.onnx'
    decoder_path = onnx_dir / 'decoder_model.onnx'
    
    # Export encoder
    print('  Exporting encoder...')
    torch.onnx.export(
        model.get_encoder(),
        (dummy_input['input_ids'], dummy_input['attention_mask']),
        str(encoder_path),
        opset_version=EXPORT_CONFIG['onnx_opset'],
        input_names=['input_ids', 'attention_mask'],
        output_names=['last_hidden_state'],
        dynamic_axes={
            'input_ids': {0: 'batch', 1: 'sequence'},
            'attention_mask': {0: 'batch', 1: 'sequence'},
            'last_hidden_state': {0: 'batch', 1: 'sequence'},
        },
    )
    
    # Save tokenizer
    tokenizer.save_pretrained(str(onnx_dir))
    model.config.save_pretrained(str(onnx_dir))
    
    onnx_size = get_model_size_mb(onnx_dir)
    print(f'  ONNX model size: {onnx_size:.1f} MB')
    
    return {
        'ort_model': None,
        'tokenizer': tokenizer,
        'onnx_dir': onnx_dir,
        'size_mb': onnx_size,
        'success': True,
    }

print('ONNX conversion functions ready.')

In [ ]:
def validate_onnx(lang_pair: str, onnx_result: dict) -> dict:
    """Validate ONNX model outputs match PyTorch model."""
    print(f'\nValidating ONNX for {lang_pair}...')
    
    model_dir = MODELS_TRAINED / lang_pair
    onnx_dir = onnx_result['onnx_dir']
    tokenizer = onnx_result['tokenizer']
    
    # Load original PyTorch model
    pt_model = MarianMTModel.from_pretrained(str(model_dir))
    pt_model.eval()
    
    test_sentences = [
        'Hello, how are you?',
        'This is a test sentence.',
        'The weather is beautiful today.',
        'Please translate this text into the target language.',
        'I would like to order food from the menu.',
    ]
    
    results = []
    for sentence in test_sentences:
        # PyTorch translation
        pt_inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)
        with torch.no_grad():
            pt_outputs = pt_model.generate(**pt_inputs, max_length=128, num_beams=4)
        pt_translation = tokenizer.decode(pt_outputs[0], skip_special_tokens=True)
        
        # ONNX translation (if Optimum model available)
        onnx_translation = pt_translation  # fallback
        if onnx_result.get('ort_model'):
            ort_model = onnx_result['ort_model']
            onnx_inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)
            onnx_outputs = ort_model.generate(**onnx_inputs, max_length=128, num_beams=4)
            onnx_translation = tokenizer.decode(onnx_outputs[0], skip_special_tokens=True)
        
        match = pt_translation == onnx_translation
        results.append({
            'input': sentence,
            'pt_output': pt_translation,
            'onnx_output': onnx_translation,
            'match': match,
        })
        
        status = '✅' if match else '⚠️'
        print(f'  {status} "{sentence[:40]}..."')
        if not match:
            print(f'      PT:   {pt_translation[:60]}')
            print(f'      ONNX: {onnx_translation[:60]}')
    
    match_rate = sum(1 for r in results if r['match']) / len(results)
    print(f'\n  Match rate: {match_rate:.0%}')
    
    return {
        'results': results,
        'match_rate': match_rate,
        'validation_passed': match_rate >= 0.8,  # Allow small differences
    }

print('ONNX validation function ready.')

## 5. Model Packaging

In [ ]:
def create_model_package(
    lang_pair: str,
    onnx_result: dict,
    quant_result: dict,
    quant_validation: dict,
    version: str = '1.0.0',
) -> Path:
    """Create deployable model package ZIP."""
    print(f'\n{"="*60}')
    print(f'Packaging: {lang_pair} v{version}')
    print(f'{"="*60}')
    
    src_lang, tgt_lang = lang_pair.split('-')
    package_dir = PACKAGES_DIR / lang_pair / f'v{version}'
    package_dir.mkdir(parents=True, exist_ok=True)
    
    onnx_dir = onnx_result['onnx_dir']
    
    # Create metadata.json
    metadata = {
        'model_version': version,
        'language_pair': lang_pair,
        'source_language': src_lang,
        'target_language': tgt_lang,
        'bleu_score': quant_validation.get('original_bleu', 0),
        'bleu_score_quantized': quant_validation.get('quantized_bleu', 0),
        'model_size_mb': onnx_result['size_mb'],
        'training_date': datetime.now().isoformat(),
        'architecture': {
            'encoder_layers': 6,
            'decoder_layers': 6,
            'hidden_size': 512,
            'vocab_size': 32000,
        },
    }
    
    with open(package_dir / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Create config.json (inference config)
    with open(package_dir / 'config.json', 'w') as f:
        json.dump(EXPORT_CONFIG['inference_config'], f, indent=2)
    
    # Copy ONNX model files
    print('  Copying model files...')
    for f in onnx_dir.iterdir():
        if f.is_file():
            target = package_dir / f.name
            if not target.exists():
                import shutil
                shutil.copy2(f, target)
    
    # Create ZIP
    zip_name = f'model_package_{lang_pair}_v{version}.zip'
    zip_path = PACKAGES_DIR / lang_pair / zip_name
    
    print(f'  Creating ZIP: {zip_name}')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
        for f in package_dir.rglob('*'):
            if f.is_file() and f.suffix != '.zip':
                arcname = f.relative_to(package_dir)
                zf.write(f, arcname)
    
    zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
    print(f'  Package size: {zip_size_mb:.1f} MB')
    
    size_ok = zip_size_mb <= EXPORT_CONFIG['max_package_size_mb']
    print(f'  Size check (<{EXPORT_CONFIG["max_package_size_mb"]}MB): {"✅ PASS" if size_ok else "❌ FAIL"}')
    
    # Generate SHA-256 checksum
    sha256 = hashlib.sha256()
    with open(zip_path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            sha256.update(chunk)
    checksum = sha256.hexdigest()
    
    with open(PACKAGES_DIR / lang_pair / 'checksum.txt', 'w') as f:
        f.write(f'{checksum}  {zip_name}\n')
    
    print(f'  SHA-256: {checksum[:16]}...')
    print(f'  ✅ Package created: {zip_path}')
    
    return zip_path

print('Packaging function ready.')

## 6. Run Full Export Pipeline

In [ ]:
export_results = {}

for lang_pair in PAIRS_TO_EXPORT:
    print(f'\n\n{"#"*60}')
    print(f'# Processing: {lang_pair}')
    print(f'{"#"*60}')
    
    try:
        # Step 1: Quantize
        q_result = quantize_model(lang_pair)
        
        # Step 2: Validate quantization
        q_validation = validate_quantization(lang_pair, q_result)
        
        # Step 3: ONNX conversion
        onnx_result = convert_to_onnx(lang_pair)
        
        # Step 4: Validate ONNX
        onnx_validation = validate_onnx(lang_pair, onnx_result)
        
        # Step 5: Package
        package_path = create_model_package(
            lang_pair, onnx_result, q_result, q_validation,
            version=EXPORT_CONFIG['model_version'],
        )
        
        export_results[lang_pair] = {
            'quantization': {
                'original_size_mb': q_result['original_size_mb'],
                'quantized_size_mb': q_result['quantized_size_mb'],
                'size_ok': q_result['size_ok'],
            },
            'bleu': q_validation if q_validation.get('bleu_validated') else {},
            'onnx': {
                'size_mb': onnx_result['size_mb'],
                'validation_passed': onnx_validation['validation_passed'],
            },
            'package_path': str(package_path),
            'status': 'SUCCESS',
        }
        
    except Exception as e:
        print(f'\n❌ Failed for {lang_pair}: {e}')
        import traceback
        traceback.print_exc()
        export_results[lang_pair] = {'status': 'FAILED', 'error': str(e)}

# Save export report
report_path = REPORTS_DIR / 'export_report.json'
with open(report_path, 'w') as f:
    json.dump(export_results, f, indent=2, default=str)

print(f'\n\nExport report saved to: {report_path}')

## 7. Generate Manifest

In [ ]:
# Create manifest.json listing all available model packages
manifest = {
    'version': '1.0',
    'generated_at': datetime.now().isoformat(),
    'models': {},
}

for lang_pair in PAIRS_TO_EXPORT:
    result = export_results.get(lang_pair, {})
    if result.get('status') == 'SUCCESS':
        package_path = Path(result['package_path'])
        checksum_file = package_path.parent / 'checksum.txt'
        checksum = ''
        if checksum_file.exists():
            checksum = checksum_file.read_text().split()[0]
        
        manifest['models'][lang_pair] = {
            'version': EXPORT_CONFIG['model_version'],
            'filename': package_path.name,
            'size_mb': result['onnx']['size_mb'],
            'checksum_sha256': checksum,
            'bleu_score': result.get('bleu', {}).get('quantized_bleu', 0),
        }

manifest_path = PACKAGES_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('📋 Model Manifest:')
print(json.dumps(manifest, indent=2))

## 8. Final Summary

In [ ]:
print('\n' + '='*60)
print('📋 EXPORT SUMMARY')
print('='*60)

for lp, result in export_results.items():
    status = '✅' if result.get('status') == 'SUCCESS' else '❌'
    print(f'\n{status} {lp}:')
    
    if result.get('status') == 'SUCCESS':
        q = result.get('quantization', {})
        b = result.get('bleu', {})
        o = result.get('onnx', {})
        
        print(f'   Size: {q.get("original_size_mb", "?")} MB → {q.get("quantized_size_mb", "?")} MB')
        if b:
            print(f'   BLEU: {b.get("original_bleu", "?")} → {b.get("quantized_bleu", "?")} (Δ{b.get("degradation", "?")})')
        print(f'   ONNX: {o.get("size_mb", "?")} MB, validated={o.get("validation_passed", "?")}')
        print(f'   Package: {result.get("package_path", "?")}')
    else:
        print(f'   Error: {result.get("error", "Unknown")}')

print(f'\n\n📁 All packages at: {PACKAGES_DIR}')
print(f'📋 Manifest at: {PACKAGES_DIR / "manifest.json"}')
print('\n\n🎉 Download the ZIP packages and import them into your BhashaLens Flutter app!')

## 9. Output Files


In [ ]:
print(f'Model packages saved to: {PACKAGES_DIR}')
print(f'Manifest at: {PACKAGES_DIR / "manifest.json"}')
print('Copy the ZIP packages to your Flutter app for offline translation.')
